In [3]:
pip install httpx selectolax jupyter

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip available: 22.3.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [9]:
import json, httpx
from selectolax.lexbor import LexborHTMLParser


In [ ]:
def fetch(url):
    r = httpx.get(
        url,
        headers={"User-Agent": "MarathonETL/0.1 (you@email.com)"},
        timeout=20,
        follow_redirects=True,
    )
    r.raise_for_status()
    return r.text

def parse_jsonld_events(html):
    events = []
    for node in LexborHTMLParser(html).css('script[type="application/ld+json"]'):
        try:
            data = json.loads(node.text())
        except json.JSONDecodeError:
            continue
        for item in (data if isinstance(data, list) else [data]):
            if item.get("@type") == "Event":
                events.append({
                    "name": item.get("name"),
                    "event_date": item.get("startDate"),
                    "city": (item.get("location", {}).get("address", {}) or {}).get("addressLocality"),
                })
    return events, data

In [ ]:
url = "https://runsignup.com/Races"   # replace with a real race page
html = fetch(url)
len(html)

122796

In [26]:
for node in LexborHTMLParser(html).css('script[type="application/ld+json"]'):
        try:
            data = json.loads(node.text())
        except json.JSONDecodeError:
            continue
        for item in (data if isinstance(data, list) else [data]):
            # if item.get("@type") == "Event":
            #     events.append({
            #         "name": item.get("name"),
            #         "event_date": item.get("startDate"),
            #         "city": (item.get("location", {}).get("address", {}) or {}).get("addressLocality"),
            #     })
            print(item)
data

{'@context': 'http://schema.org', '@type': 'Organization', 'name': 'RunSignup', 'url': 'https://runsignup.com', 'logo': 'https://d368g9lw5ileu7.cloudfront.net/img/nonversioned/jsonLdLogo2024.png', 'sameAs': ['https://www.facebook.com/runsignup']}
{'@context': 'http://schema.org', '@type': 'WebSite', 'url': 'https://runsignup.com', 'potentialAction': {'@type': 'SearchAction', 'target': 'https://runsignup.com/Races?s=&name={q}', 'query-input': 'required name=q'}}


{'@context': 'http://schema.org',
 '@type': 'WebSite',
 'url': 'https://runsignup.com',
 'potentialAction': {'@type': 'SearchAction',
  'target': 'https://runsignup.com/Races?s=&name={q}',
  'query-input': 'required name=q'}}

In [27]:
"Boston" in html   # use a race name you can see on the page

False